<a href="https://colab.research.google.com/github/zbeyza/Advanced-Functional-EDA/blob/main/supervised_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install required packages
%pip install "transformers>=4.36.0" "trl>=0.7.0" "datasets>=2.14.0" "torch>=2.0.0"
%pip install "accelerate>=0.24.0" "peft>=0.7.0" "trackio"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 11.8 MB/s  0:00:00


### exploring SmolLM3's chat templates

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
import json
from typing import Optional, Dict, Any

if torch.cuda.is_available():
  device = "cuda"
  print(f"using CUDA GPU {torch.cuda.get_device_name(0)}")
  print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
elif hasattr(torch.backends, "mps") and torch.backend.mps.is_available():
  device = "mps"
  print("Using Apple MPS")
else:
  device = "cpu"
  print("Using CPU - need GPU to train models")

# authenticate with huggingface
from huggingface_hub import login

using CUDA GPU Tesla T4
GPU memory: 15.6GB


In [ ]:
# loading SmolLM3 models

# base and instruct model for comparison
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

# tokenizers
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# models
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

print("models loaded successfully")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

models loaded successfully


In [ ]:
# chat template formatting
# create different types of conversations to test

conversations = {
    "simple_qa": [
        {"role": "user", "content": "what is machine learning?"}
    ],

    "with_system": [
        {"role": "system", "content": "you are a helpful AI assistantspecialized in explaining technical concepts clearly."},
        {"role": "user", "content": "what is machien learninf?"}
    ],

    "multi_turn": [
        {"role": "system", "content": "you are a math ttor."},
        {"role": "user", "content": "what is calculus?"},
        {"role": "asssitant", "content": "calculus is a branch of mathematics that deals with rates of change and accumulation of quantities."},
        {"role": "user", "content": "can you give me a simple example?"}
    ],

    "reasoning_task": [
        {"role": "user", "content": "solve step by step: if a train travels 120 miles in 2 hours, what is its average speed?"}
    ]
}

for conv_type, messages in conversations.items():
  print(f"--- {conv_type.upper()} ---")

  # format without gneration prompt (for completed conversations)
  formatted_complete = instruct_tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=False
  )

  # format with generation prompt (for inference)
  formatted_prompt = instruct_tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  print("complete conversation format")
  print(formatted_complete)
  print("\nWith genration prompt:")
  print(formatted_prompt)
  print("\n" + "="*60 + "\n")

--- SIMPLE_QA ---
complete conversation format
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 23 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current s

In [ ]:
# base vs instruct model responses

# same prompt on both models
test_prompt = "explain quantum computing in simple terms."

# prepare the prompt for base model (no chat tempalte)
base_inputs = base_tokenizer(test_prompt, return_tensors="pt").to(device)

# prepare the prompr for instruct model (with caht template)
instruct_messages = [{"role": "user",
                      "content": test_prompt}]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages,
    tokenize=False,
    add_generation_prompt=True
)
instruct_inputs = instruct_tokenizer(instruct_formatted, return_tensors="pt").to(device)

# generate responses
print("=== Model compariosn ===\n")

print("Base Model Response:")
with torch.no_grad():
  base_outputs = base_model.generate(
      **base_inputs,
      max_new_tokens=150,
      temperature=0.7,
      do_sample=True,
      pad_token_id=base_tokenizer.eos_token_id
  )
  base_response = base_tokenizer.decode(base_outputs[0],
                                        skip_special_tokens=True)
  print(base_response[len(test_prompt):]) # show only the generated part

print("\n" + "="*60)
print("Instruct Model Response:")
with torch.no_grad():
  instruct_outputs = instruct_model.generate(
      **instruct_inputs,
      max_new_tokens=150,
      temperature=0.7,
      do_sample=True,
      pad_token_id=instruct_tokenizer.eos_token_id
  )
  instruct_response = instruct_tokenizer.decode(instruct_outputs[0],
                                                skip_special_tokens=True)

  # extract only the asssitant's response
  assistant_start = instruct_response.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
  assistant_response = instruct_response[assistant_start:].split("<|im_end|>")[0]
  print(assistant_response)

"""
the diference between the base model and the instruct model
the base model continues the string, while the instruct model uses the caht template
"""

=== Model compariosn ===

Base Model Response:
 I've read a bit about it, but the explanations seemed to be too technical and difficult to understand.
I'm a software developer, and I'm looking for a way to explain quantum computing to my friends who are not computer science majors. Ideally, I would like to do this in a simple way that they can understand without requiring a lot of technical knowledge.
If you've done this before, can you please share the resources you used, or the methods you used to explain it?
Thanks in advance for any help you can provide.
I'm glad you asked! In this article, I will explain the basics of quantum computing in a way that is easy to understand for everyone, even those without a technical background. So, let's dive in!


Instruct Model Response:
nowledge Cutoff Date: June 2025
Today Date: 23 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involve

'\nthe diference between the base model adn the instruct model\nthe base model continues the string, while the instruct model uses the caht template\n'

In [ ]:
# test dual-mode reasoning

#test SmolLM3 reasoning capabilities
reasoning_prompts = [
    "what is 15 x 24? show youe work.",
    "a recepie calls for 2 cups of flour for 12 cookies. how much flour is needed for 30 cookies?"
    "if I have $50 adn spend $18.75 on lunch adn $12.30 on book, how much money do I have left?"
]

print("=== Testing Reasoning Capabilities ===\n")

for i, prompt in enumerate(reasoning_prompts, 1):
  print(f"Problem {i}: {prompt}")

  messages = [{"role": "user",
               "content": prompt}]
  formatted_prompt = instruct_tokenizer.apply_chat_template(
      messages, tokenize=False, add_genration_prompt=True
  )
  inputs = instruct_tokenizer(formatted_prompt, return_tensors="pt").to(device)

  with torch.no_grad():
    outputs = instruct_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,
        do_sample=True,
        pad_token_id=instruct_tokenizer.eos_token_id
    )
    response = instruct_tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_start = response.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
    assistant_response = response[assistant_start:].split("|im_end|>")[0]
    print(f"Answer: {assistant_response}")

print("\n" + "-"*70 + "\n")

"""

"""


=== Testing Reasoning Capabilities ===

Problem 1: what is 15 x 24? show youe work.
Answer: nowledge Cutoff Date: June 2025
Today Date: 23 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of t

'\n\n'

### dataset processing for SFT

In [ ]:
# explore the SmolTalk2 dataset

# load adn explore the SmolTalk2 dataset
#print("=== Exploring SmolTalk2 Dataset ===\n")

# load the SFT subset
#dataset_dict = load_dataset("HuggingFaceTB/smoltalk2", "SFT")
#print(f"total splits: {len(dataset_dict)}")
#print(f"available splits: {list(dataset_dict.keys())}")
#print(f"number of total rows: {sum([dataset_dict[d].num_rows for d in dataset_dict])}")
#print(f"dataset structure: {dataset_dict}")

#---------------
from datasets import load_dataset_builder

print("=== EXPLORING SMOLTALK2 DATASET ===\n")

# Reads only the metadata in the dataset card; no data is downloaded
builder = load_dataset_builder("HuggingFaceTB/smoltalk2", "SFT")
splits = builder.info.splits
features = list(builder.info.features)

print(f"Total splits: {len(splits)}")
print(f"Available splits: {list(splits.keys())}")
print(f"Number of total rows: {sum(s.num_examples for s in splits.values())}")

print("Dataset structure: DatasetDict({")
for name, info in splits.items():
    print(f"    {name}: Dataset({{")
    print(f"        features: {features},")
    print(f"        num_rows: {info.num_examples}")
    print("    })")
print("})")


=== EXPLORING SMOLTALK2 DATASET ===



README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Total splits: 25
Available splits: ['LongAlign_64k_Qwen3_32B_yarn_131k_think', 'OpenThoughts3_1.2M_think', 'aya_dataset_Qwen3_32B_think', 'multi_turn_reasoning_if_think', 's1k_1.1_think', 'smolagents_toolcalling_traces_think', 'smoltalk_multilingual8_Qwen3_32B_think', 'smoltalk_systemchats_Qwen3_32B_think', 'table_gpt_Qwen3_32B_think', 'LongAlign_64k_context_lang_annotated_lang_6_no_think', 'Mixture_of_Thoughts_science_no_think', 'OpenHermes_2.5_no_think', 'OpenThoughts3_1.2M_no_think_no_think', 'hermes_function_calling_v1_no_think', 'smoltalk_multilingual_8languages_lang_5_no_think', 'smoltalk_smollm3_everyday_conversations_no_think', 'smoltalk_smollm3_explore_instruct_rewriting_no_think', 'smoltalk_smollm3_smol_magpie_ultra_no_think', 'smoltalk_smollm3_smol_rewrite_no_think', 'smoltalk_smollm3_smol_summarize_no_think', 'smoltalk_smollm3_systemchats_30k_no_think', 'table_gpt_no_think', 'tulu_3_sft_personas_instruction_following_no_think', 'xlam_traces_no_think', 'smoltalk_everyday_con

In [ ]:
# process different dataset types

"""
the smoltalk2 dataset is a clooection of open-source datasets complied together for convenience.
- it contains a mixture of useful post training use casesi like tool usei long context adn more.
- they are all in caht formati which is easy to use for training
"""

# function to process different dataset fromats
def process_qa_dataset(examples, question_col, answer_col):
  """
  Process Q&A datasets into chat formats
  """
  processed = []

  for question, answer in zip(examples[question_col], examples[answer_col]):
    messages = [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer}
    ]
    processed.append(messages)

  return {"messages": processed}

def process_instruction_dataset(examples):
  """
  Process instruction-following datasets
  """
  processed = []

  for instruction, response in zip(examples["instruction"], examples["response"]):
    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": response}
    ]
    processed.append(messages)

  return {"messages": processed}

# example: process GSM8K math dataset
print("=== Processing GSM8K Dataset ===\n")

gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]") # small subset for demo
print(f"original gsm8k exampel: {gsm8k[0]}")

# convert to chat format
def process_gsm8k(examples):
  processed = []
  for question, answer in zip(examples["question"], examples["answer"]):
    messages = [
        {"role": "system", "content": "you are a math tutor. solveproblems step by step."},
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer}
    ]
    processed.append(messages)
  return {"messages": processed}

gsm8k_processed = gsm8k.map(process_gsm8k, batched=True, remove_columns=gsm8k.column_names)
print(f"processed exampel: {gsm8k_processed[0]}")




=== Processing GSM8K Dataset ===



README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

original gsm8k exampel: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

processed exampel: {'messages': [{'content': 'you are a math tutor. solveproblems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


In [ ]:
# apply chat templates to the dataset

# function to aplpty caht templates to processed datasets
def apply_chat_template_to_dataset(dataset, tokenizer):
  """Apply chat template to dataset for training"""

  def format_messages(examples):
    formatted_texts =  []

    for messages in examples["messages"]:
      # apply chat template
      formatted_text = tokenizer.apply_chat_template(
          messages,
          tokenize=False,
          add_generation_prompt=False # we want the complete conversation
      )
      formatted_texts.append(formatted_text)

    return {"text": formatted_texts}

  return dataset.map(format_messages, batched=True)

# apply to processed gsm8k dataset
gsm8k_formatted = apply_chat_template_to_dataset(gsm8k_processed, instruct_tokenizer)
print("=== FORMATTED TRAINING DATA ===")
print(gsm8k_formatted[0]["text"])


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

=== FORMATTED TRAINING DATA ===
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 26 November 2026
Reasoning Mode: /think

## Custom Instructions

you are a math tutor. solveproblems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

